# Show Your Work: Chain-of-Thought Reasoning and Automatic Prompt Optimisation

---

> ### Assignment
> You will take an LLM that answers grade-school maths word problems **in one
> shot** and make it a **careful reasoner** - first by changing *how* it is asked
> to think (**chain-of-thought**), then by letting an optimiser, rather than you,
> decide *what* the prompt should say (**prompt optimisation**).
>
> **Part 1: Chain-of-Thought** holds the task fixed and varies the reasoning
> format: **direct answer → zero-shot CoT → few-shot CoT → Plan-and-Solve →
> self-consistency**. Every technique is
> scored on the same harness, including its **token cost**.
>
> **Part 2: Prompt Optimisation** treats the prompt as a parameter to be searched.
> You measure how **brittle** hand-written prompts are, run an **APE → OPRO**
> instruction search and **exemplar selection** (kNN and bootstrapped demos) on a
> **dev set**, and only then touch the **held-out test set** - and a second
> dataset (**SVAMP**) to check the optimised prompt transfers.
>
> **Cells marked `# -- PROVIDED --` carry no marks** - just run them.
> **Cells marked `### CODE HERE ###` carry marks** - these are yours to complete.
> Reflection cells (`***your answer here***`) are part of your grade where a mark
> value is shown.
>
> Use `seed = 2026` wherever a seed is accepted. State the seed, `EVAL_N` and
> `DEV_N` you actually ran with in your final reflection.

---

## What You Will Build

| Section | Approach | Key Idea |
|---------|----------|----------|
| 3 | Direct vs zero-shot CoT | "Let's think step by step" - reasoning tokens as a scratchpad |
| 4 | Few-shot CoT | Worked solutions as exemplars teach the *shape* of a good derivation |
| 5 | Plan-and-Solve | Extract variables and plan before calculating - targets skipped steps |
| 6 | Self-consistency | Sample several chains, vote; agreement as a confidence signal |
| 8 | Prompt sensitivity | How much does wording / exemplar order alone move accuracy? |
| 9 | APE → OPRO | Let an LLM propose instructions; iterate with a scored history |
| 10 | Exemplar optimisation | kNN-retrieved demos; bootstrapped (model-written) demos |
| 11 | Held-out evaluation + transfer | Test once on GSM8K test, then on SVAMP |

**Total Marks: 100**  (Part 1: 50 | Part 2: 50)

### Datasets
- **GSM8K** (`openai/gsm8k`, config `main`) - 8.5K grade-school maths word
  problems. Every answer field is a **human-written worked solution** ending in
  `#### <number>`, so each example gives us both a **gold number** (for scoring)
  and a **gold rationale** (for few-shot CoT exemplars).
- **SVAMP** (`ChilleD/SVAMP`) - one- and two-step arithmetic word problems built
  by perturbing simpler datasets; used as a **transfer set** for the optimised
  prompt in §11.

### Three disjoint slices - and why
| Slice | Split | Used for | Touched in |
|-------|-------|----------|------------|
| `eval_set` (50) | GSM8K **test** | scoring every technique | Part 1, §11 only |
| `dev_set` (25) | GSM8K **train** | scoring candidate prompts during optimisation | §8-§10 |
| `exemplar_pool`, `boot_pool`, `knn_pool` | GSM8K **train** | exemplars / demos | §4, §10 |

In Part 2, **every search decision is made on `dev_set`.** Choosing a prompt by
its test score and then reporting that test score is the prompt-engineering
version of training on the test set.

### The Central Question
> *A 3B model can usually do every individual step of a four-step word problem,
> yet often gets the final number wrong when forced to reply in one shot. How much
> of that gap closes when it is allowed to write its reasoning down - and once it
> reasons, can an automated search find a better prompt than the one you would
> write by hand?*

---

# PART 1: Chain-of-Thought Reasoning  *(50 marks)*

The task, model and scoring are fixed. Only the **reasoning format** the prompt
asks for changes from section to section, so every difference in accuracy (and
in tokens spent) can be attributed to the technique.

## Setup  *(provided - run all cells, no marks)*

We use **Llama-3.2-3B-Instruct** through the HuggingFace `pipeline`, a small
instruction-tuned model that runs on a free Colab GPU. A GPU runtime is strongly recommended
(Colab: *Runtime → Change runtime type → T4 GPU*). This assignment makes a few
thousand generations, so the setup adds a **batched** helper, `chat_batch`, and
the evaluation harness uses it for you.

In [ ]:
# -- PROVIDED --------------------------------------------------------------
# Run once. -q keeps the output quiet.
%pip install -q transformers accelerate datasets sentence-transformers

In [ ]:
# -- PROVIDED --------------------------------------------------------------
import os, re, sys, json, math, random, string, warnings
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

SEED = 2026
random.seed(SEED)
np.random.seed(SEED)
print("Imports ready.")

In [ ]:
# -- PROVIDED --------------------------------------------------------------
import torch
from transformers import pipeline, GenerationConfig, set_seed
set_seed(SEED)   # makes sampled generations (self-consistency, APE, OPRO) repeatable

# Non-gated mirror of Llama-3.2-3B-Instruct (no HF token required).
# If you have gated access you may swap in "meta-llama/Llama-3.2-3B-Instruct".
MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"

pipe = pipeline(
    model=MODEL_ID,
    torch_dtype=torch.bfloat16,   # use torch.float16 on a T4 if bf16 is slow
    device_map="auto",
)

tok = pipe.tokenizer
tok.clean_up_tokenization_spaces = False
if tok.pad_token_id is None:
    tok.pad_token_id = tok.eos_token_id
tok.padding_side = "left"          # required for batched generation with a decoder-only model

pipe.model.generation_config = GenerationConfig.from_model_config(pipe.model.config)
pipe.model.generation_config.pad_token_id = tok.eos_token_id

print("Loaded", MODEL_ID, "on", pipe.model.device)

In [ ]:
# -- PROVIDED --------------------------------------------------------------
BATCH_SIZE = 8   # lower to 4 if you run out of GPU memory

def chat_batch(prompts, max_new_tokens=384, temperature=0.0, batch_size=BATCH_SIZE):
    """prompts: list of (system_prompt, user_prompt) tuples.
    Greedy if temperature == 0, else nucleus sampling (top_p=0.95).
    Returns the assistant replies as a list of strings, in the same order."""
    convs = [[{"role": "system", "content": s}, {"role": "user", "content": u}]
             for s, u in prompts]
    do_sample = bool(temperature and temperature > 0)
    outs = pipe(
        convs,
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=(temperature if do_sample else None),
        top_p=(0.95 if do_sample else None),
        batch_size=batch_size,
        generation_config=None,
    )
    return [o[0]["generated_text"][-1]["content"].strip() for o in outs]

def chat(user_prompt, system_prompt="You are a helpful assistant.",
         max_new_tokens=384, temperature=0.0):
    """Single-turn convenience wrapper around chat_batch."""
    return chat_batch([(system_prompt, user_prompt)], max_new_tokens, temperature, batch_size=1)[0]

def n_tokens(text):
    """Number of tokens in a generated reply - our measure of inference cost."""
    return len(tok.encode(text or "", add_special_tokens=False))

def fmt_num(x):
    """72.0 -> '72', 2.5 -> '2.5' (for writing gold answers into prompts)."""
    x = float(x)
    return str(int(x)) if x.is_integer() else str(round(x, 4))

# Smoke test (no task yet):
print(chat("In one sentence, what is chain-of-thought prompting?"))

## 1. Data Loading and Understanding  <font color="red">**[6 marks]**</font>

### 1.1 Load GSM8K and build the working sets  <font color="red">[3 marks]</font>

**What to do:**
- Load `openai/gsm8k`, config `"main"`, with `load_dataset`.
- `eval_set` = `EVAL_N = 50` examples from the **test** split
  (`.shuffle(seed=SEED).select(...)`).
- Shuffle the **train** split **once** with `seed=SEED` into `train_shuf`, then cut
  four **disjoint** slices from it, in this order:
  `exemplar_pool` = rows `[0, 8)`, `dev_set` = the next `DEV_N = 25`,
  `boot_pool` = the next `BOOT_N = 20`, and `knn_pool` = `KNN_POOL_N = 1000` rows
  starting at row `100`.
- Each GSM8K example has `question` and `answer`. The `answer` field is a worked
  solution whose last line is `#### <number>`, and it contains calculator
  annotations like `<<3*4=12>>`. Write three helpers:
  - `gold_number(example)` → the float after `####` (remove thousands commas).
  - `gold_rationale(example)` → the solution text **before** `####`, with every
    `<<...>>` annotation removed and surrounding whitespace stripped.
  - `n_steps(example)` → number of non-empty lines in `gold_rationale` (a cheap
    proxy for how many reasoning steps the problem needs).

In [ ]:
EVAL_N, DEV_N, BOOT_N, KNN_POOL_N = 50, 25, 20, 1000
from datasets import load_dataset

gsm = ### CODE HERE ###                          # load_dataset(..., "main")
eval_set   = ### CODE HERE ###                   # 50 shuffled TEST examples
train_shuf = ### CODE HERE ###                   # train split shuffled once with SEED

exemplar_pool = ### CODE HERE ###                # rows [0, 8)
dev_set       = ### CODE HERE ###                # next DEV_N rows
boot_pool     = ### CODE HERE ###                # next BOOT_N rows
knn_pool      = ### CODE HERE ###                # KNN_POOL_N rows starting at 100

def gold_number(example):
    """Float after '####' in example['answer'] (commas removed)."""
    ### CODE HERE ###

def gold_rationale(example):
    """Worked solution before '####', with <<...>> calculator annotations removed."""
    ### CODE HERE ###

def n_steps(example):
    """Number of non-empty lines in the gold rationale."""
    ### CODE HERE ###

# Sanity checks (these should pass):
assert gold_number({"answer": "Some working.\n#### 1,234"}) == 1234.0
assert gold_rationale({"answer": "She has 3*4=<<3*4=12>>12 eggs.\n#### 12"}) == "She has 3*4=12 eggs."
assert n_steps({"answer": "Step one.\nStep two.\n\nStep three.\n#### 3"}) == 3
q = lambda ds: set(ds["question"])
assert not (q(exemplar_pool) & q(dev_set)) and not (q(dev_set) & q(boot_pool)) and not (q(dev_set) & q(knn_pool)), "slices overlap!"

print("eval:", len(eval_set), "| exemplar pool:", len(exemplar_pool), "| dev:", len(dev_set),
      "| boot pool:", len(boot_pool), "| knn pool:", len(knn_pool))
print("gold number of eval[0]:", gold_number(eval_set[0]))

### 1.2 Understand the data  <font color="red">[3 marks]</font>

**What to do:** on the evaluation set, produce all of the following:
- A bar chart of **`n_steps`** per question. Then add a column `bucket` that maps
  steps to `"1-2"`, `"3"`, `"4"`, `"5+"` and print the count per bucket - the
  harness uses these same buckets to break accuracy down by difficulty.
- A histogram of **question length in words**, with mean and median printed.
- The share of gold answers that are **integers**, and the min / max gold answer.
- **One full example**: the question, the raw `answer` field, the cleaned
  `gold_rationale`, and the `gold_number` - so you can see what an exemplar will
  look like in §4.

In [ ]:
### CODE HERE ###
# 1) bar chart of n_steps; bucket counts ("1-2", "3", "4", "5+")
# 2) question-length histogram + mean/median
# 3) share of integer gold answers; min / max gold answer
# 4) print one full example (question, raw answer, cleaned rationale, gold number)

### Interpretation Checkpoint  *(self-review, not scored)*

**[Your Answer]:** Answer each question in 2-4 sentences.

1. GSM8K answers are single numbers. Why does that make scoring more reliable than comparing free-text answers, and what new failure does it introduce that you would not have if you simply compared strings?
2. The gold rationales are written by humans and are terse. Predict: when you use them as few-shot exemplars in §4, will the model's reasoning get longer or shorter than in zero-shot CoT? Why?
3. Why must `dev_set` come from the **train** split and never overlap `eval_set`, given that Part 2 will try dozens of prompts on it?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

## 2. The Evaluation Harness  <font color="red">**[6 marks]**</font>

Every technique ends in a number, so a technique is scored by **numeric
accuracy**: parse a number out of the model's reply and compare it to the gold
number. Parsing is where most silent bugs live - a reply that says
`"... so the total is $1,250.00.\nAnswer: $1,250"` must parse to `1250.0`, not
to `0.0` or `None`. We also log two diagnostics for every run:

- **Parse-fail rate** - replies from which no number could be extracted. A
  technique with high accuracy *and* a high parse-fail rate is leaving marks on
  the table through formatting alone.
- **Mean output tokens** - the inference cost of the technique. CoT is not free;
  in §7 you will plot accuracy against cost.

### 2.1 Implement the answer parser and the metric  <font color="red">[6 marks]</font>

**What to do:** implement the two functions below.
- `extract_number(text)` - if the text contains `Answer:` (any case), look only at
  what follows the **last** `Answer:`, and return the **first** number there;
  otherwise return the **last** number anywhere in the text. Handle `$`,
  thousands commas, decimals, negatives and a trailing full stop. Return a
  `float`, or `None` if there is no number (or `text` is `None`).
- `numeric_match(pred, gold, tol=1e-4)` - `1.0` if `pred` is not `None` and
  `|pred - gold| <= tol`, else `0.0`.

In [ ]:
def extract_number(text):
    """Number after the last 'Answer:' if present, else the last number in the text."""
    ### CODE HERE ###

def numeric_match(pred, gold, tol=1e-4):
    ### CODE HERE ###

# Sanity checks (these should pass):
assert extract_number("... so she pays $1,250.00.\nAnswer: $1,250") == 1250.0
assert extract_number("Answer: 18 eggs") == 18.0
assert extract_number("5 + 3 = 8. Then 8 * 2 = 16.") == 16.0
assert extract_number("Answer: -3.5") == -3.5
assert extract_number("Step 1: 4 apples. Answer: 12. (Checked in step 2.)") == 12.0
assert extract_number("I am not sure.") is None and extract_number(None) is None
assert numeric_match(72.0, 72) == 1.0 and numeric_match(None, 5) == 0.0 and numeric_match(71.0, 72) == 0.0
print("Parser checks passed.")

In [ ]:
# -- PROVIDED --------------------------------------------------------------
# One harness for EVERY technique. You write make_prompt(example) -> (system, user);
# the harness batches the generations, parses, scores, and stores everything.
RESULTS = {}   # name -> {"acc", "parse_fail", "tokens", "preds", "raws", "correct"}

def bucket_of(example):
    s = n_steps(example)
    return "1-2" if s <= 2 else ("3" if s == 3 else ("4" if s == 4 else "5+"))

def record_result(name, examples, preds, tokens_per_q, raws=None):
    """Store a run in RESULTS. Also used directly by self-consistency (Section 6)."""
    golds = [gold_number(ex) for ex in examples]
    correct = [numeric_match(p, g) for p, g in zip(preds, golds)]
    RESULTS[name] = {
        "acc": 100 * np.mean(correct),
        "parse_fail": 100 * np.mean([p is None for p in preds]),
        "tokens": float(np.mean(tokens_per_q)),
        "preds": preds, "raws": raws, "correct": correct,
    }
    return RESULTS[name]

def evaluate(make_prompt, examples, name, max_new_tokens=384, parse_fn=None,
             temperature=0.0, record=True, verbose_every=10):
    """Run make_prompt over examples, parse with parse_fn (default extract_number),
    score by numeric_match. record=False returns the result without storing it
    (used for scoring candidate prompts on dev_set in Part 2)."""
    parse_fn = parse_fn or extract_number
    raws = chat_batch([make_prompt(ex) for ex in examples],
                      max_new_tokens=max_new_tokens, temperature=temperature)
    preds = [parse_fn(r) for r in raws]
    toks = [n_tokens(r) for r in raws]
    if verbose_every:
        for i in range(0, len(examples), verbose_every):
            print(f"  [{i:>3}/{len(examples)}] pred={preds[i]!r:<10} gold={gold_number(examples[i])!r}")
    if record:
        res = record_result(name, examples, preds, toks, raws)
    else:
        correct = [numeric_match(p, gold_number(ex)) for p, ex in zip(preds, examples)]
        res = {"acc": 100 * np.mean(correct), "parse_fail": 100 * np.mean([p is None for p in preds]),
               "tokens": float(np.mean(toks)), "preds": preds, "raws": raws, "correct": correct}
    print(f"==> {name}:  Acc = {res['acc']:.1f}   parse-fail = {res['parse_fail']:.1f}%   "
          f"mean out-tokens = {res['tokens']:.0f}")
    return res

def accuracy_by_steps(names, examples=None):
    """Accuracy (%) per n_steps bucket for the given RESULTS entries."""
    examples = examples if examples is not None else eval_set
    buckets = [bucket_of(ex) for ex in examples]
    rows = {}
    for name in names:
        c = RESULTS[name]["correct"]
        rows[name] = {b: 100 * np.mean([ci for ci, bi in zip(c, buckets) if bi == b])
                      for b in ["1-2", "3", "4", "5+"] if b in buckets}
    df = pd.DataFrame(rows).T.round(1)
    df.columns = [f"steps {c} (n={buckets.count(c)})" for c in df.columns]
    return df

print("Harness ready. RESULTS dict initialised.")

## 3. Direct Answer vs Zero-Shot Chain-of-Thought  <font color="red">**[8 marks]**</font>

### Why reasoning tokens help

A transformer does a fixed amount of computation per generated token. If the
prompt demands `Answer: <number>` immediately, every intermediate quantity of a
four-step problem has to be computed *inside* the forward pass that emits the
first digit. **Zero-shot chain-of-thought** (Kojima et al., 2022) adds one
sentence - *"Let's think step by step."* - and the model writes intermediate
results into its own context, where later tokens can read them. The price is
paid in output tokens, and the reply is now a paragraph, so the prompt **must**
still pin a final `Answer: <number>` line for the parser.

### 3.1 Direct answer (no reasoning)  <font color="red">[3 marks]</font>

**What to do:** write `DIRECT_SYSTEM` and `make_prompt_direct(example)` that
instruct the model to reply with **only** `Answer: <number>` - no working. Run
`evaluate(...)` with a **small** `max_new_tokens` (e.g. 16) under the name
`"Direct answer"`.

In [ ]:
DIRECT_SYSTEM = ### CODE HERE ###      # maths assistant; reply ONLY "Answer: <number>"

def make_prompt_direct(example):
    ### CODE HERE ###                  # return (system, user)

_ = evaluate(make_prompt_direct, eval_set, "Direct answer", max_new_tokens=16)

### 3.2 Zero-shot CoT  <font color="red">[3 marks]</font>

**What to do:** write `COT_SYSTEM` and `make_prompt_zs_cot(example)`: the user
prompt contains the question, then *"Let's think step by step."*, then an
instruction to finish with a final line `Answer: <number>`. Print **one raw
trace** first to check the model reasons and your parser grabs the right number,
then evaluate with `max_new_tokens=384` as `"Zero-shot CoT"`.

Finally, call the provided `accuracy_by_steps(["Direct answer", "Zero-shot CoT"])`.

In [ ]:
COT_SYSTEM = ### CODE HERE ###         # careful maths reasoner; must end with "Answer: <number>"

def make_prompt_zs_cot(example):
    ### CODE HERE ###

# Peek at one raw reasoning trace before scoring:
### CODE HERE ###

_ = evaluate(make_prompt_zs_cot, eval_set, "Zero-shot CoT", max_new_tokens=384)
display(accuracy_by_steps(["Direct answer", "Zero-shot CoT"]))

### 3.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report Direct vs Zero-shot CoT accuracy **and** mean output tokens. Using your `accuracy_by_steps` table, does the CoT gain grow with the number of steps? Explain why that pattern (or its absence) is what the "extra computation" argument predicts.
2. Find one CoT trace where the reasoning is correct but the parsed answer is wrong, **or** where the answer is right but a step in the reasoning is wrong. Quote the relevant lines. What does this say about treating the written chain as an explanation of how the model got its answer?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 4. Few-Shot Chain-of-Thought  <font color="red">**[8 marks]**</font>

### Few-shot CoT

Wei et al. (2022) introduced CoT as a *few-shot* technique: each in-context
exemplar shows a question, a **worked solution**, and the answer. The model
imitates the shape of the derivation - how finely to split steps, how to label
quantities, where to stop. GSM8K gives us gold rationales for free.

To see what the *rationales* contribute (as opposed to the exemplars' format),
you will build the same exemplar block **with** and **without** the solution
text and compare both against zero-shot CoT.

### 4.1 Build, run, and score few-shot prompts  <font color="red">[6 marks]</font>

**What to do:**
- `to_demo(example)` → `{"question", "rationale", "answer"}` using
  `gold_rationale` and `fmt_num(gold_number(...))`. (Part 2 builds demos from other
  sources - kNN neighbours, the model's own traces - in this same dict shape.)
- `build_exemplar_block(demos, with_rationale=True)` → the demos rendered as
  ```
  Question: <question>
  Solution: <rationale>        # omitted when with_rationale=False
  Answer: <answer>
  ```
  separated by blank lines.
- `make_prompt_fs(example, demos, with_rationale)` → the block, then the target
  `Question: ...` followed by `Solution:` (or `Answer:` when
  `with_rationale=False`) so the model continues the pattern. Use `COT_SYSTEM` for
  the CoT version and `DIRECT_SYSTEM` for the answer-only version.
- Use `FS_K = 4` demos: the **first 4** of `exemplar_pool`. Evaluate as
  `"Few-shot direct"` (`max_new_tokens=16`) and `"Few-shot CoT"` (`384`).

In [ ]:
FS_K = 4

def to_demo(example):
    ### CODE HERE ###

def build_exemplar_block(demos, with_rationale=True):
    ### CODE HERE ###

def make_prompt_fs(example, demos, with_rationale=True):
    ### CODE HERE ###

FS_DEMOS = [to_demo(exemplar_pool[i]) for i in range(FS_K)]
blk = build_exemplar_block(FS_DEMOS[:2])
assert blk.count("Question:") == 2 and "Solution:" in blk
assert "Solution:" not in build_exemplar_block(FS_DEMOS[:2], with_rationale=False)
print(blk)

_ = evaluate(lambda ex: make_prompt_fs(ex, FS_DEMOS, with_rationale=False), eval_set, "Few-shot direct", max_new_tokens=16)
_ = evaluate(lambda ex: make_prompt_fs(ex, FS_DEMOS, with_rationale=True),  eval_set, "Few-shot CoT",    max_new_tokens=384)

### 4.2 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Put Direct, Few-shot direct, Zero-shot CoT and Few-shot CoT side by side. Which comparison isolates the effect of the *rationales* (as opposed to the exemplars' format), and how big is it in your run?
2. Compare mean output tokens of Zero-shot CoT and Few-shot CoT. Did the model copy the terse style of the GSM8K rationales? Was that good or bad for accuracy - and does it match the prediction you made in the §1 checkpoint?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 5. Plan-and-Solve Prompting  <font color="red">**[8 marks]**</font>

### Plan first, then calculate

Zero-shot CoT failures on GSM8K fall into roughly three bins: **missing steps**,
**arithmetic slips**, and **misread quantities**. *"Let's think step by step"*
says nothing about *how* to structure the steps, so the model often starts
calculating before it has worked out what the question needs - and silently
drops a step.

**Plan-and-Solve (PS+)** (Wang et al., 2023) replaces the trigger with an
instruction to work in two phases:

1. **Understand and plan** - extract the relevant variables and their values,
   and devise a plan that lists the sub-goals in order;
2. **Solve** - carry out the plan step by step, computing intermediate variables
   *"paying attention to correct calculation and commonsense"*.

It targets **missing steps** and **misread quantities** by making the model
commit to *what* it will compute before it computes anything. Two things are
worth checking rather than assuming: whether the model actually **follows** the
plan-first instruction, and whether the questions PS fixes are the ones
zero-shot CoT got wrong for the reason PS is meant to address.

### 5.1 Build, run, and score Plan-and-Solve  <font color="red">[4 marks]</font>

**What to do:**
- Write `PS_INSTRUCTION` - your own wording of the PS+ trigger (identify the
  variables and their values → make a plan → execute it step by step with careful
  arithmetic) - and `make_prompt_ps(example)` that uses it **in place of**
  *"Let's think step by step."*, keeping `COT_SYSTEM` and the `Answer:` line
  unchanged. Evaluate as `"Plan-and-Solve"`.
- **Plan adherence.** Write `has_plan(trace)` → `True` iff the trace contains at
  least one `=` and the word *plan* (any case, as a whole word) appears **before
  the first `=`** - a cheap check that a plan was stated before calculating
  started. Report the share of PS traces with a plan, and PS accuracy on traces
  **with** vs **without** one (with the count in each group).
- Display `accuracy_by_steps(["Zero-shot CoT", "Plan-and-Solve"])`.

In [ ]:
PS_INSTRUCTION = ### CODE HERE ###

def make_prompt_ps(example):
    ### CODE HERE ###

def has_plan(trace):
    """True iff the trace contains '=' and the word 'plan' appears before the first '='."""
    ### CODE HERE ###

assert has_plan("Plan: 1) find the total. 2) divide.\nTotal = 3*4 = 12") is True
assert has_plan("Total = 3*4 = 12. That was the plan.") is False
assert has_plan("Let me plan this out carefully.") is False      # no calculation at all
assert has_plan("Planet count: 8 = 8") is False                  # 'Planet' is not the word 'plan'

_ = evaluate(make_prompt_ps, eval_set, "Plan-and-Solve", max_new_tokens=448)

### CODE HERE ###
# share of PS traces with has_plan == True; PS accuracy with vs without a plan (+ group sizes)
display(accuracy_by_steps(["Zero-shot CoT", "Plan-and-Solve"]))

### 5.2 Where do Zero-shot CoT and Plan-and-Solve disagree?  <font color="red">[2 marks]</font>

**What to do:** from `RESULTS[...]["correct"]`, build the **2×2 table** of
outcomes for Zero-shot CoT vs Plan-and-Solve (both right / only ZS-CoT right /
only PS right / both wrong) and display it. Then print **2 questions PS fixed**
and **1 question PS broke** (if any), each with the gold number and the two
traces side by side, so you can see what changed in the reasoning.

In [ ]:
### CODE HERE ###
# 1) 2x2 contingency table (ZS-CoT correct? x PS correct?) as a DataFrame
# 2) print 2 questions only PS got right and 1 only ZS-CoT got right, with both traces

### 5.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report PS against Zero-shot CoT (accuracy and mean tokens) together with your 2×2 table. For the questions PS fixed, was it the plan that made the difference (a step ZS-CoT skipped, or a quantity it misread), or something else? Quote the evidence.
2. Using your plan-adherence numbers: did the model actually state a plan before calculating, and was accuracy higher when it did? What does that tell you about whether PS works through the *content* of the instruction or simply by making the model write more?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 6. Self-Consistency  <font color="red">**[8 marks]**</font>

### Self-consistency, and agreement as confidence

A single greedy chain commits to its first wrong turn. **Self-consistency**
(Wang et al., 2023) samples $n$ chains at temperature $>0$, parses each answer,
and returns the **majority vote**. Numeric answers make voting clean: two chains
that reach `72` agree even if their wording differs.

The vote also yields a free **confidence score**: the *agreement*, i.e. the share
of the $n$ chains that back the winner. If agreement is informative, accuracy on
high-agreement questions should be clearly higher than on low-agreement ones -
which is exactly what you would need to decide when to trust the model or
escalate to a human.

Because the samples are independent, you can also read off accuracy for smaller
$n$ **without new generations** by voting over the first $m$ samples.

### 6.1 Build, run, and score self-consistency  <font color="red">[6 marks]</font>

**What to do:**
- `majority_vote(nums)` → `(winner, agreement)`. Ignore `None` when counting
  votes but **keep it in the denominator** (a chain that produced no answer did not
  support the winner); treat numbers equal to 4 decimal places as the same vote;
  break ties by **first-seen**; return `(None, 0.0)` if every sample is `None`.
- `sc_sample(make_prompt, examples, n, temperature)` → `(samples, tokens)` where
  `samples[i]` is the list of `n` parsed answers for `examples[i]` and `tokens[i]`
  is the **total** output tokens spent on that question. *Tip:* repeat each prompt
  `n` times in one `chat_batch` call.
- Run it with `make_prompt_zs_cot`, `SC_N = 5`, `SC_TEMP = 0.7`. Then, for
  `m ∈ {1, 3, 5}`, vote over the first `m` samples and store each run with
  `record_result` as `"Self-consistency (n=m)"` (tokens scale with `m`).
- Append each question's `n=5` agreement to `SC_AGREEMENT`, then report the
  **mean agreement**, the number of **unanimous** questions, and accuracy on
  questions with agreement `≥ 0.8` vs `< 0.8` (with the count in each group).

In [ ]:
SC_N, SC_TEMP = 5, 0.7
SC_AGREEMENT = []   # -- PROVIDED -- one agreement score per eval question (n = SC_N)

def majority_vote(nums):
    ### CODE HERE ###

assert majority_vote([3.0, 5.0, 3.0, None, 5.0]) == (3.0, 0.4)   # tie 3 vs 5 -> first seen
assert majority_vote([None, None]) == (None, 0.0)
assert majority_vote([7.0, 7.00001, 2.0]) == (7.0, 2/3)

def sc_sample(make_prompt, examples, n=SC_N, temperature=SC_TEMP, max_new_tokens=384):
    ### CODE HERE ###

samples, sc_tokens = sc_sample(make_prompt_zs_cot, eval_set)

### CODE HERE ###
# 1. for m in [1, 3, 5]: vote over samples[i][:m]; record_result(f"Self-consistency (n={m})", ...)
#    tokens for m samples ~ sc_tokens[i] * m / SC_N
# 2. fill SC_AGREEMENT with the n=5 agreement of each question
# 3. mean agreement; # unanimous; accuracy for agreement >= 0.8 vs < 0.8 (with counts)

### 6.2 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report accuracy for n = 1, 3, 5 against greedy Zero-shot CoT, with mean agreement and unanimous count. Is the n=1 sampled run worse than greedy? Why would that be expected, and why does voting recover (or not recover) the loss?
2. Using your high- vs low-agreement split, is agreement a useful confidence signal? Describe how you would use it in a deployment that can afford to send 20% of questions to a human.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 7. Part 1 Comparison  <font color="red">**[6 marks]**</font>

### 7.1 Accuracy-vs-cost table and chart  <font color="red">[3 marks]</font>

**What to do:**
- Build a DataFrame from `RESULTS` with one row per Part-1 technique (Direct,
  Few-shot direct, Zero-shot CoT, Few-shot CoT, Plan-and-Solve,
  Self-consistency n=1/3/5) and columns `Acc`, `Parse-fail %`, `Tokens`. Sort by
  `Acc` descending and display it.
- Draw a **scatter plot of accuracy (y) against mean output tokens (x)**, one
  labelled point per technique, and mark the techniques on the **Pareto
  frontier** (no other technique is both cheaper and at least as accurate).

In [ ]:
### CODE HERE ###
# 1) DataFrame from RESULTS (Part-1 techniques), columns Acc / Parse-fail % / Tokens, sorted by Acc
# 2) accuracy-vs-tokens scatter with labels; highlight Pareto-optimal techniques

### 7.2 Error taxonomy  <font color="red">[3 marks]</font>

**What to do:** take your best **single-chain** text-CoT technique (Zero-shot CoT,
Few-shot CoT or Plan-and-Solve - whichever scored highest), and print **6 questions
it got wrong** with the gold number, the prediction, and the full trace. Label each
by hand in a dict `ERROR_LABELS = {index: category}` with one of
`"arithmetic slip"`, `"wrong plan / missing step"`, `"misread question"`,
`"format / parse"`, and plot the counts as a bar chart. End with a one-line
takeaway: *which error type would a better prompt fix, and which would it not?*

In [ ]:
### CODE HERE ###
# pick the best single-chain text-CoT technique; print 6 wrong examples (Q / gold / pred / trace)
ERROR_LABELS = {
    # eval index: "arithmetic slip" | "wrong plan / missing step" | "misread question" | "format / parse"
}
# bar chart of label counts + one-line takeaway

### Part 1 Reflection  *(self-review, not scored)*

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Across Part 1, what was the single biggest jump in accuracy, and was it bought with tokens (CoT, voting) or with structure (exemplars, planning)?
2. State, in one sentence with your own numbers, the best *single-chain* CoT accuracy - the baseline that Part 2's optimised prompt must beat.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

---
# PART 2: Prompt Optimisation  *(50 marks)*

In Part 1 **you** chose every word of every prompt. Part 2 treats the prompt as a
**parameter** and searches over it, in the same loop used to tune any other
hyperparameter:

> **propose** candidate prompts → **score** each on `dev_set` → **select** → repeat

Two things are searched separately:
1. the **instruction** - the reasoning trigger that follows the question
   (§8-§9: hand-written variants, then APE and OPRO), and
2. the **exemplars** - which worked solutions appear in the prompt and who wrote
   them (§10: fixed, kNN-retrieved, or bootstrapped from the model's own traces).

Two rules hold throughout Part 2:
- **The output contract never changes.** Every candidate prompt ends with the same
  `FORMAT_LINE`, so we are optimising *reasoning*, not the parser.
- **Selection happens on `dev_set` only.** `eval_set` is used exactly once, in §11.

## 8. How Brittle Is a Hand-Written Prompt?  <font color="red">**[8 marks]**</font>

### Sensitivity, and the noise floor

Before searching, we need to know two things: how much accuracy moves when you
make changes that *should not matter* (rewording, reordering exemplars), and how
much it moves by **chance** on a 25-question dev set. For a dev accuracy $p$
measured on $n$ questions, the binomial standard error is

$$\text{SE} = 100\cdot\sqrt{\frac{p\,(1-p)}{n}} \quad\text{(in accuracy points)}$$

With $n = 25$ a single question is worth 4 points, and the SE near $p = 0.6$ is
about 10 points. Any "improvement" found by the optimiser must be read against
this number.

### 8.1 Instruction sensitivity  <font color="red">[4 marks]</font>

**What to do:**
- Extend `INSTRUCTION_VARIANTS` with **at least 3 instructions of your own**
  (6+ in total). Make them genuinely different (tone, length, what they ask the
  model to do first).
- `make_prompt_from_instruction(instruction)` → returns a `make_prompt(example)`
  **function** whose user prompt is `Question: <q>` + blank line + `instruction`
  + newline + `FORMAT_LINE`, with `COT_SYSTEM` as the system prompt.
- `score_instruction(instruction, source)` → evaluate on `dev_set` with
  `record=False`, append `{"instruction", "dev_acc", "source"}` to the provided
  `PROMPT_BANK`, and return the dev accuracy. Score every variant with
  `source="human"`.
- Display the variants sorted by dev accuracy and print the **range**, the **std**
  of the scores, and the **binomial SE** at the mean accuracy.

In [ ]:
# -- PROVIDED --
FORMAT_LINE = "End your response with a final line of the form 'Answer: <number>'."
PROMPT_BANK = []   # list of {"instruction", "dev_acc", "source"} - every prompt you score on dev

INSTRUCTION_VARIANTS = [
    "Let's think step by step.",                                                         # Kojima et al., 2022
    "Let's work this out in a step by step way to be sure we have the right answer.",   # APE (Zhou et al., 2023)
    "Take a deep breath and work on this problem step-by-step.",                         # OPRO (Yang et al., 2024)
    ### CODE HERE ###   # add at least 3 of your own
]

def make_prompt_from_instruction(instruction):
    ### CODE HERE ###

def score_instruction(instruction, source):
    ### CODE HERE ###

for ins in INSTRUCTION_VARIANTS:
    score_instruction(ins, source="human")

### CODE HERE ###   # sorted table; range, std, binomial SE at the mean dev accuracy

### 8.2 Exemplar-order sensitivity  <font color="red">[2 marks]</font>

**What to do:** take the 4 demos `FS_DEMOS` from §4 and build **3 orderings**: the
original, plus 2 shuffles with `random.Random(SEED + i).shuffle`. Score few-shot CoT
(`make_prompt_fs`) with each ordering on `dev_set` (`record=False`) and report the
three accuracies and their range. Keep the original-order score as
`DEV_ACC_RANDOM_DEMOS` - it is the "fixed exemplars" baseline in §10.

In [ ]:
### CODE HERE ###
# 3 orderings of FS_DEMOS -> dev accuracy each -> range
DEV_ACC_RANDOM_DEMOS = ### CODE HERE ###

### 8.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Compare the spread across instruction variants and across exemplar orders with the binomial SE you computed. How much of the spread could be chance? What does that imply for how you should read a "+4 points" result from the optimiser in §9?
2. Suppose you had picked the best of your instruction variants by its score on `eval_set` and reported that score. Explain precisely why the reported number would be optimistic, and in which direction the error grows as you try more variants.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 9. Automatic Instruction Optimisation: APE → OPRO  <font color="red">**[14 marks]**</font>

### Using the LLM as the optimiser

**APE** (Automatic Prompt Engineer; Zhou et al., 2023) *proposes* instructions by
showing an LLM some input-output demonstrations and asking it to infer the
instruction that produced them - a one-shot, "reverse-engineer the task" step.

**OPRO** (Optimisation by PROmpting; Yang et al., 2024) makes the search
*iterative*. Each step shows the LLM a **meta-prompt** containing the best
instructions found so far **with their scores, sorted from worst to best**, plus a
couple of task examples, and asks for a new instruction that scores higher. The
scored history plays the role of a gradient: the optimiser sees what worked.

```
PROMPT_BANK ──► top instructions + scores ──► meta-prompt ──► LLM (T≈1) ──► new candidates
     ▲                                                                          │
     └──────────────────── score each on dev_set ◄──────────────────────────────┘
```

Here the same 3B model is both the *optimiser* and the *scorer*. The papers used
far stronger optimiser models, so treat what you find as a small-scale replication.

### 9.1 APE: propose instructions from demonstrations  <font color="red">[4 marks]</font>

**What to do:**
- `build_ape_meta_prompt(demos)` → a user prompt in the APE "forward generation"
  style: *"I gave a friend an instruction and some math questions. The friend read
  the instruction and wrote a solution for every question. Here are the
  question-solution pairs: … The instruction was:"*, followed by a request to
  write **only** the instruction, in one sentence. Use the first 3 of `FS_DEMOS`.
- `clean_instruction(text)` → first non-empty line, with surrounding quotes and any
  leading `"Instruction:"` label removed, truncated to 300 characters.
- Sample `APE_K = 5` candidates at `temperature=0.9` (one `chat_batch` call with the
  meta-prompt repeated), clean them, drop duplicates of anything already in
  `PROMPT_BANK`, and score the rest with `score_instruction(..., source="ape")`.

In [ ]:
APE_K = 5

def build_ape_meta_prompt(demos):
    ### CODE HERE ###

def clean_instruction(text):
    ### CODE HERE ###

assert clean_instruction('\n  Instruction: "Solve it carefully."\nextra line') == "Solve it carefully."

### CODE HERE ###
# sample APE_K candidates -> clean -> dedupe against PROMPT_BANK -> score_instruction(..., "ape")

### 9.2 OPRO: iterate with a scored history  <font color="red">[6 marks]</font>

**What to do:**
- `build_opro_meta_prompt(bank, top=8, examples=...)` → a meta-prompt that
  1. states the task (write an instruction that is placed after a grade-school
     maths question to help a model solve it),
  2. lists the `top` highest-scoring instructions in `bank`, **sorted in ascending
     order of `dev_acc`** (best last), each as
     `text:\n<INS>...</INS>\nscore:\n<int>`,
  3. shows 2 example problems (question + gold answer) from `exemplar_pool[4:6]`,
  4. asks for **one** new instruction, different from all above, that achieves a
     higher score, enclosed in `<INS>` and `</INS>`.
- `parse_ins(text)` → the content of the **first** `<INS>...</INS>` pair, stripped;
  if there are no tags, fall back to `clean_instruction(text)`.
- Run `OPRO_STEPS = 3` steps. At each step build the meta-prompt from the
  **current** bank, sample `OPRO_CANDS = 2` candidates at `temperature=1.0`, parse,
  skip duplicates, and score with `source=f"opro-{step}"`. Record every scored
  candidate in `OPRO_TRAJECTORY` as `(step, instruction, dev_acc)`.

In [ ]:
OPRO_STEPS, OPRO_CANDS = 3, 2
OPRO_TRAJECTORY = []   # -- PROVIDED -- (step, instruction, dev_acc)

def build_opro_meta_prompt(bank, top=8, examples=None):
    ### CODE HERE ###

def parse_ins(text):
    ### CODE HERE ###

assert parse_ins("Sure! <INS>Break it into parts.</INS> <INS>other</INS>") == "Break it into parts."
assert parse_ins("Solve the problem slowly.") == "Solve the problem slowly."

print(build_opro_meta_prompt(PROMPT_BANK)[:1500])   # inspect your meta-prompt once

### CODE HERE ###
# for step in 1..OPRO_STEPS: meta-prompt from current bank -> OPRO_CANDS samples -> parse -> dedupe -> score

### 9.3 Optimisation trajectory  <font color="red">[2 marks]</font>

**What to do:** plot dev accuracy of every scored instruction against its "step"
(human = step 0, APE = step 0.5, OPRO = 1..3) as a scatter, and overlay the
**best-so-far** line. Draw a horizontal band of ± 1 SE (from §8.1) around the best
human instruction. Print the **top-3 instructions** in `PROMPT_BANK` with their
source and dev accuracy, and set `BEST_INSTRUCTION` to the top one (ties → the one
scored first).

In [ ]:
### CODE HERE ###
# scatter of dev_acc by step + best-so-far line + ±1 SE band around the best human prompt
# print top-3 instructions (source, dev_acc)
BEST_INSTRUCTION = ### CODE HERE ###

### 9.4 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Did APE or OPRO find an instruction that beats the best human-written one on dev? Is the margin larger than one SE? Quote the winning instruction and say whether it reads like something a human would have written.
2. The OPRO meta-prompt sorts instructions from worst to best and shows scores. Why might ordering and showing the scores matter to the optimiser LLM? Name one change to your meta-prompt that you would expect to make the search work better, and why.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 10. Exemplar Optimisation: kNN and Bootstrapped Demos  <font color="red">**[12 marks]**</font>

### Which demos, and who wrote them?

Your few-shot CoT in §4 used the same 4 exemplars for every question. Two ideas
from the literature change that:

- **kNN exemplar selection** (Liu et al., 2022): embed every candidate question,
  and for each test question pick the $k$ **most similar** training questions as
  its demos. The demos then share structure (rates, percentages, unit
  conversions) with the question at hand. Place the most similar demo **last**,
  right before the test question.
- **Bootstrapped demos** (the idea behind DSPy's `BootstrapFewShot`; Khattab et
  al., 2024): run the model itself on some training questions, **keep only the
  traces that reach the correct answer**, and use those model-written traces as
  demos. The demos are then written in the model's own style instead of the
  terse human style of GSM8K.

### 10.1 kNN-selected exemplars  <font color="red">[5 marks]</font>

**What to do:**
- Load `SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")` and encode
  every `knn_pool` question with `normalize_embeddings=True` into `knn_emb`.
- `knn_demos(question, k=FS_K)` → encode the question, take the `k` pool questions
  with the highest cosine similarity, and return them as demos (`to_demo`) in
  **ascending** order of similarity (most similar last).
- Score few-shot CoT with per-question kNN demos on `dev_set` (`record=False`),
  store the accuracy as `DEV_ACC_KNN_DEMOS`, and compare it with
  `DEV_ACC_RANDOM_DEMOS`. Print the demos chosen for one dev question to check
  they are topically similar, and report the **mean top-1 similarity** over the dev
  set.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = ### CODE HERE ###
knn_emb  = ### CODE HERE ###          # (KNN_POOL_N, 384), L2-normalised

def knn_demos(question, k=FS_K):
    ### CODE HERE ###

### CODE HERE ###
# inspect demos for one dev question; mean top-1 similarity over dev_set
DEV_ACC_KNN_DEMOS = ### CODE HERE ###

### 10.2 Bootstrapped (model-written) demos  <font color="red">[5 marks]</font>

**What to do:**
- Run `make_prompt_from_instruction(BEST_INSTRUCTION)` greedily on `boot_pool`
  (`record=False`). Keep the examples whose parsed answer is **correct**, and report
  the **bootstrap yield** (correct / `BOOT_N`).
- Turn the first `FS_K` correct traces into demos: `rationale` = the trace with its
  final `Answer:` line removed; `answer` = the gold number. If fewer than `FS_K`
  traces were correct, use what you have and say so.
- Score few-shot CoT with these `BOOT_DEMOS` on `dev_set` → `DEV_ACC_BOOT_DEMOS`.
- Print one bootstrapped demo next to the human `gold_rationale` for the same
  question.

In [ ]:
### CODE HERE ###
# 1. run BEST_INSTRUCTION on boot_pool; keep correct traces; print yield
# 2. BOOT_DEMOS = first FS_K correct traces as demos (strip the final 'Answer:' line)
# 3. dev accuracy with BOOT_DEMOS
# 4. print one bootstrapped rationale vs the human gold rationale
BOOT_DEMOS = ### CODE HERE ###
DEV_ACC_BOOT_DEMOS = ### CODE HERE ###

display(pd.DataFrame({
    "Exemplar strategy": ["fixed human demos (§8.2)", "kNN human demos", "bootstrapped model demos"],
    "Dev Acc": [DEV_ACC_RANDOM_DEMOS, DEV_ACC_KNN_DEMOS, DEV_ACC_BOOT_DEMOS],
}))

### 10.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Rank the three exemplar strategies by dev accuracy and put the differences next to your SE. Did retrieving *similar* questions help more or less than changing *who wrote* the solution?
2. Bootstrapping keeps a trace only if its final number is right. Why can a kept trace still be a bad demo? How would you filter more strictly without human labelling?

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 11. Held-Out Evaluation and Transfer  <font color="red">**[14 marks]**</font>

### Opening the test set - once

Everything in §8-§10 was chosen on 25 dev questions. Two questions remain:
1. **Does it hold up on unseen GSM8K questions?** A gap between dev and test
   accuracy measures how much the search fitted the dev set (plus ordinary
   sampling noise).
2. **Does it transfer?** SVAMP problems are shorter and built by *perturbing*
   simpler problems (changing the question asked, swapping the order of facts,
   adding irrelevant numbers). An instruction tuned on GSM8K may or may not carry
   over.

### 11.1 Final prompt on the GSM8K test set  <font color="red">[6 marks]</font>

**What to do:**
- `FINAL_DEMO_STRATEGY` = the best of `{"none", "fixed", "knn", "boot"}` by dev
  accuracy, where `"none"` means zero-shot with `BEST_INSTRUCTION` (its dev
  accuracy is in `PROMPT_BANK`).
- `make_prompt_optimised(example)` → the chosen demo block (if any), then
  `Question: <q>`, a blank line, `BEST_INSTRUCTION`, and `FORMAT_LINE`.
- For a like-for-like baseline, use `make_prompt_from_instruction("Let's think step
  by step.")`. Evaluate **both** on `eval_set` as `"Baseline CoT (test)"` and
  `"Optimised (test)"`.
- Display a table with dev accuracy, test accuracy, **dev→test gap** and mean
  tokens for both prompts, plus the number of test questions that **flipped** in
  each direction (baseline wrong → optimised right, and baseline right →
  optimised wrong).

In [ ]:
FINAL_DEMO_STRATEGY = ### CODE HERE ###

def make_prompt_optimised(example):
    ### CODE HERE ###

_ = evaluate(make_prompt_from_instruction("Let's think step by step."), eval_set, "Baseline CoT (test)")
_ = evaluate(make_prompt_optimised, eval_set, "Optimised (test)")

### CODE HERE ###
# table: dev acc, test acc, dev->test gap, tokens for both prompts; flip counts in each direction

### 11.2 Transfer to SVAMP  <font color="red">[6 marks]</font>

**What to do:**
- Load `ChilleD/SVAMP` (inspect `.column_names` and the available splits first)
  and take `SVAMP_N = 50` shuffled test examples (seed `SEED`).
- `to_gsm_format(row)` → `{"question": Body + " " + Question, "answer": "#### <Answer>"}`,
  so that `gold_number`, `evaluate` and every `make_prompt` work unchanged.
- Evaluate the baseline and the optimised prompt on `svamp_set` as
  `"Baseline CoT (SVAMP)"` and `"Optimised (SVAMP)"`. (If your optimised prompt uses
  kNN demos, they still come from the **GSM8K** `knn_pool` - note that in your
  reflection.)
- Print one question each prompt got right that the other got wrong.

In [ ]:
SVAMP_N = 50
svamp_raw = ### CODE HERE ###
svamp_set = ### CODE HERE ###         # list of GSM8K-shaped dicts

def to_gsm_format(row):
    ### CODE HERE ###

_ = evaluate(make_prompt_from_instruction("Let's think step by step."), svamp_set, "Baseline CoT (SVAMP)")
_ = evaluate(make_prompt_optimised, svamp_set, "Optimised (SVAMP)")
### CODE HERE ###   # one question each prompt got right and the other got wrong

### 11.3 Reflection  <font color="red">[2 marks]</font>

**[Your Answer]:** Answer each question in 2-4 sentences.

1. Report dev and test accuracy for both prompts. Was the optimised prompt's dev advantage preserved on test? Use the flip counts to judge whether the test difference is meaningful or could be a handful of coin-flips.
2. Did the optimised prompt help, hurt, or do nothing on SVAMP? Give one reason, specific to what you optimised (instruction wording, demo choice), why a prompt tuned on GSM8K might not transfer.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

## 12. Final Comparison  <font color="red">**[2 marks]**</font>

### 12.1 Master table and chart  <font color="red">[2 marks]</font>

**What to do:** build one table of the headline runs - **Direct answer**,
**Zero-shot CoT**, the **best Part-1 technique**, **Baseline CoT (test)**,
**Optimised (test)**, **Baseline CoT (SVAMP)** and **Optimised (SVAMP)** - with
`Acc` and `Tokens`. Plot accuracy as bars with mean tokens on a secondary axis (or
annotated on each bar). This is the "one-shot answer → reasoned answer →
optimised prompt" story in one picture.

In [ ]:
### CODE HERE ###
# summary DataFrame across the headline runs + bar chart (accuracy) with token cost shown

## Final Reflection  *(self-review, not scored)*

**[Your Answer]:** Answer each question in 2-4 sentences.

1. State, with your own numbers, the full arc: direct answer → best CoT technique → optimised prompt (test). Which step bought the most accuracy per extra token?
2. Self-consistency and prompt optimisation both spend extra compute - one at **inference** time on every question, the other **once** at design time. For a system answering a million questions a day, which would you invest in first and why?
3. Your optimiser and scorer were the same 3B model. What would you expect to change if the optimiser were a much stronger model but the scorer stayed the same?
4. Name one risk of automatically optimised prompts that hand-written prompts do not have (think about interpretability, dev-set size, and what happens when the model is upgraded).
5. Report the `SEED`, `EVAL_N`, `DEV_N` and any other parameter you changed from the defaults, and why.

*Write your answers here:*

1. ***your answer here***

2. ***your answer here***

3. ***your answer here***

4. ***your answer here***

5. ***your answer here***

---
## Mark Summary

| Section | Topic | What you write | Marks |
|---------|-------|----------------|-------|
| **Part 1** | **Chain-of-Thought Reasoning** | | **50** |
| Setup | Model + batched chat helper | *Provided - run only* | 0 |
| 1 | Data loading + understanding (GSM8K, disjoint slices) | Code | 6 |
| 2 | Evaluation harness (answer parser + numeric match) | Code | 6 |
| 3 | Direct vs zero-shot CoT (build + run + reflect) | Code + answers | 8 |
| 4 | Few-shot CoT, with vs without rationales | Code + answers | 8 |
| 5 | Plan-and-Solve (adherence + disagreement analysis) | Code + answers | 8 |
| 6 | Self-consistency + agreement as confidence | Code + answers | 8 |
| 7 | Accuracy-vs-cost comparison + error taxonomy | Code | 6 |
| **Part 2** | **Prompt Optimisation** | | **50** |
| 8 | Prompt sensitivity + noise floor | Code + answers | 8 |
| 9 | APE → OPRO instruction search | Code + answers | 14 |
| 10 | Exemplar optimisation (kNN, bootstrapped) | Code + answers | 12 |
| 11 | Held-out test + SVAMP transfer | Code + answers | 14 |
| 12 | Final comparison | Code | 2 |
| | **Total** | | **100** |

---
*End of Assignment*